# Fish2 Raphe Explorer — student / Google Colab notebook

This notebook is a lightweight teaching version of the Fish2 raphe workflow.

It does four things:

1. Connects to the Fish2 neuPrint dataset.
2. Lists all neuPrint ROIs and automatically selects the ones containing `Raphe_Superior`.
3. Fetches neurons touching those ROIs, prints their body IDs, and plots their skeletons in 3D.
4. Loads the **selected-raphe clustering assignments from a previous local run**, extracts **cluster 7**, prints its body IDs, and plots those neurons too.

## Notes for students

- A neuron can **touch** a Raphe ROI with an axon or dendrite while its soma is elsewhere.
- The cluster-7 section uses a previously computed morphology clustering result; it does **not** rerun NBLAST.
- neuPrint authentication is requested securely at runtime. Do not paste tokens into notebooks that will be committed to GitHub.

In [ ]:
# ============================================================
# 0. Install dependencies
# ============================================================

import sys
import subprocess
import importlib.util

REQUIRED_PACKAGES = {
    "neuprint": "neuprint-python",
    "pandas": "pandas",
    "numpy": "numpy",
    "plotly": "plotly",
    "tqdm": "tqdm",
}

missing = [
    pip_name
    for import_name, pip_name in REQUIRED_PACKAGES.items()
    if importlib.util.find_spec(import_name) is None
]

if missing:
    print("Installing:", missing)
    subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", *missing])
else:
    print("All required packages already installed.")

In [ ]:
# ============================================================
# 1. Imports and notebook configuration
# ============================================================

from pathlib import Path
from getpass import getpass
import os

import numpy as np
import pandas as pd
import plotly.graph_objects as go

from neuprint import Client, fetch_neurons, NeuronCriteria as NC
from tqdm.auto import tqdm
from IPython.display import display

NEUPRINT_SERVER = "neuprint-fish2.janelia.org"
NEUPRINT_DATASET = "fish2"

# To keep the teaching notebook responsive, the touching-neuron skeleton plot
# defaults to a subset. Set to None to attempt all fetched neurons.
MAX_TOUCHING_SKELETONS_TO_PLOT = 100

# Cluster 7 can usually be plotted in full.
MAX_CLUSTER7_SKELETONS_TO_PLOT = None

TOUCHING_LINE_WIDTH = 1.2
CLUSTER7_LINE_WIDTH = 2.2

TOUCHING_COLOR = "rgb(80, 140, 220)"
CLUSTER7_COLOR = "rgb(235, 140, 35)"

CACHE_DIR = Path("student_cache")
SKELETON_CACHE_DIR = CACHE_DIR / "skeletons"
SKELETON_CACHE_DIR.mkdir(parents=True, exist_ok=True)

print("Cache directory:", SKELETON_CACHE_DIR.resolve())

In [ ]:
# ============================================================
# 2. Connect securely to Fish2 neuPrint
# ============================================================

token = (
    os.environ.get("NEUPRINT_APPLICATION_CREDENTIALS")
    or os.environ.get("NEUPRINT_TOKEN")
)

if not token:
    token = getpass(
        "Paste your Fish2 neuPrint token (input is hidden; it will not be printed): "
    ).strip()

if not token:
    raise RuntimeError("No neuPrint token was provided.")

c = Client(
    NEUPRINT_SERVER,
    dataset=NEUPRINT_DATASET,
    token=token,
)

print("Connected successfully.")
print("Server:", c.server)
print("Dataset:", c.dataset)

In [ ]:
# ============================================================
# 3. List all neuPrint ROIs and select Raphe superior ROIs
# ============================================================

all_rois = sorted(c.meta["roiInfo"].keys())

roi_table = pd.DataFrame({
    "roi": all_rois,
    "contains_raphe": ["raphe" in roi.lower() for roi in all_rois],
    "contains_raphe_superior": [
        ("raphe" in roi.lower() and "superior" in roi.lower())
        for roi in all_rois
    ],
})

print(f"Total ROIs in Fish2 neuPrint: {len(all_rois):,}")
display(roi_table)

raphe_superior_rois = [
    roi
    for roi in all_rois
    if "raphe" in roi.lower() and "superior" in roi.lower()
]

print("\nRaphe superior ROI names:")
for roi in raphe_superior_rois:
    print("  ", roi)

if not raphe_superior_rois:
    raise RuntimeError("No ROI names containing both 'Raphe' and 'Superior' were found.")

In [ ]:
# ============================================================
# 4. Fetch neurons touching any Raphe superior ROI
# ============================================================

raphe_superior_neurons, raphe_superior_roi_counts = fetch_neurons(
    NC(rois=raphe_superior_rois, roi_req="any"),
    client=c,
)

raphe_superior_neurons = (
    raphe_superior_neurons
    .copy()
    .drop_duplicates("bodyId")
    .sort_values("bodyId")
    .reset_index(drop=True)
)

raphe_superior_body_ids = (
    raphe_superior_neurons["bodyId"]
    .astype(int)
    .tolist()
)

print(
    f"Fetched {len(raphe_superior_body_ids):,} neurons touching at least one "
    f"Raphe superior ROI."
)

display(raphe_superior_neurons.head(20))

print("\nBody IDs:")
print(raphe_superior_body_ids)

touching_ids_path = CACHE_DIR / "raphe_superior_touching_bodyIds.csv"
pd.DataFrame({"bodyId": raphe_superior_body_ids}).to_csv(touching_ids_path, index=False)
print("\nSaved:", touching_ids_path)

In [ ]:
# ============================================================
# 5. Skeleton helpers
# ============================================================

def fetch_or_load_skeleton(body_id, heal=True):
    """
    Fetch one neuPrint skeleton and cache it as SWC.
    """
    body_id = int(body_id)
    swc_path = SKELETON_CACHE_DIR / f"{body_id}.swc"

    if not swc_path.exists():
        try:
            c.fetch_skeleton(
                body_id,
                heal=heal,
                export_path=str(swc_path),
            )
        except Exception as exc:
            print(f"Could not fetch skeleton for {body_id}: {exc}")
            return None

    rows = []
    with open(swc_path, "r", errors="ignore") as f:
        for line in f:
            line = line.strip()
            if not line or line.startswith("#"):
                continue

            parts = line.split()
            if len(parts) < 7:
                continue

            try:
                rows.append([
                    int(float(parts[0])),
                    int(float(parts[1])),
                    float(parts[2]),
                    float(parts[3]),
                    float(parts[4]),
                    float(parts[5]),
                    int(float(parts[6])),
                ])
            except ValueError:
                continue

    if not rows:
        return None

    return pd.DataFrame(
        rows,
        columns=[
            "node_id", "type", "x", "y", "z", "radius", "parent_id"
        ],
    )


def swc_line_coordinates(swc):
    """
    Convert SWC parent-child edges into Plotly line arrays.
    """
    if swc is None or swc.empty:
        return [], [], []

    xyz_by_id = {
        int(row.node_id): (float(row.x), float(row.y), float(row.z))
        for row in swc.itertuples()
    }

    xs, ys, zs = [], [], []

    for row in swc.itertuples():
        child_id = int(row.node_id)
        parent_id = int(row.parent_id)

        if parent_id < 0 or parent_id not in xyz_by_id:
            continue

        x1, y1, z1 = xyz_by_id[parent_id]
        x2, y2, z2 = xyz_by_id[child_id]

        xs.extend([x1, x2, None])
        ys.extend([y1, y2, None])
        zs.extend([z1, z2, None])

    return xs, ys, zs


def plot_skeletons_3d(
    body_ids,
    title,
    color="rgb(80, 140, 220)",
    line_width=1.2,
    max_neurons=None,
):
    """
    Fetch/cache skeletons and plot them interactively in 3D.
    """
    body_ids = [int(x) for x in body_ids]

    if max_neurons is not None and len(body_ids) > int(max_neurons):
        print(
            f"Plotting the first {int(max_neurons):,} of {len(body_ids):,} neurons "
            "for responsiveness."
        )
        plot_ids = body_ids[:int(max_neurons)]
    else:
        plot_ids = body_ids

    fig = go.Figure()
    fetched_ids = []

    for body_id in tqdm(plot_ids, desc="Loading skeletons"):
        swc = fetch_or_load_skeleton(body_id)
        if swc is None:
            continue

        xs, ys, zs = swc_line_coordinates(swc)
        if not xs:
            continue

        fig.add_trace(
            go.Scatter3d(
                x=xs,
                y=ys,
                z=zs,
                mode="lines",
                line=dict(color=color, width=line_width),
                name=str(body_id),
                hoverinfo="name",
                showlegend=False,
            )
        )
        fetched_ids.append(body_id)

    fig.update_layout(
        title=f"{title}<br><sup>n={len(fetched_ids):,} skeletons plotted</sup>",
        scene=dict(
            xaxis=dict(visible=False),
            yaxis=dict(visible=False),
            zaxis=dict(visible=False),
            aspectmode="data",
            bgcolor="white",
        ),
        paper_bgcolor="white",
        margin=dict(l=0, r=0, t=70, b=0),
        height=800,
    )

    fig.show()
    return fig, fetched_ids

In [ ]:
# ============================================================
# 6. Plot Raphe-superior-touching neuron skeletons
# ============================================================

touching_skeleton_figure, touching_skeleton_ids_plotted = plot_skeletons_3d(
    raphe_superior_body_ids,
    title="Neurons touching Raphe superior ROIs",
    color=TOUCHING_COLOR,
    line_width=TOUCHING_LINE_WIDTH,
    max_neurons=MAX_TOUCHING_SKELETONS_TO_PLOT,
)

## Load the previously identified selected-Raphe cluster 7

This section intentionally **does not rerun NBLAST**.

It searches for a previously saved assignment table containing `bodyId` and `cluster`. The notebook checks common output paths from the morphology project.

For Google Colab, you can:

- place/copy the assignment CSV under `data/`;
- mount Google Drive and set `CLUSTER_ASSIGNMENTS_PATH` manually;
- upload the assignment CSV when prompted by the fallback cell.

The cluster numbering must correspond to the selected-Raphe clustering run in which **cluster 7** was identified.

In [ ]:
# ============================================================
# 7. Locate the previous selected-Raphe assignment table
# ============================================================

# Best option: set this manually to the exact CSV from your chosen run.
CLUSTER_ASSIGNMENTS_PATH = None

SELECTED_RAPHE_CLUSTER_ID = 7

candidate_assignment_paths = [
    Path("data/ACTIVE_selected_raphe_assignments.csv"),
    Path("data/selected_raphe_assignments.csv"),
]

recursive_candidates = []
for pattern in [
    "**/ACTIVE_selected_raphe_assignments.csv",
    "**/*selected*raphe*assignments*.csv",
    "**/*cluster*assignments*.csv",
]:
    recursive_candidates.extend(Path(".").glob(pattern))

all_candidates = []
for p in candidate_assignment_paths + recursive_candidates:
    if p.exists() and p.is_file():
        rp = p.resolve()
        if rp not in all_candidates:
            all_candidates.append(rp)

if CLUSTER_ASSIGNMENTS_PATH is not None:
    selected_assignment_path = Path(CLUSTER_ASSIGNMENTS_PATH)
elif all_candidates:
    all_candidates = sorted(
        all_candidates,
        key=lambda p: (
            0 if p.name == "ACTIVE_selected_raphe_assignments.csv" else 1,
            len(str(p)),
        ),
    )
    selected_assignment_path = all_candidates[0]
else:
    selected_assignment_path = None

print("Assignment CSV candidates found:")
for p in all_candidates:
    print("  ", p)

if selected_assignment_path is not None:
    print("\nUsing:", selected_assignment_path)
else:
    print(
        "\nNo assignment CSV was found automatically. "
        "Run the next fallback cell in Colab, or set CLUSTER_ASSIGNMENTS_PATH manually."
    )

In [ ]:
# ============================================================
# 8. Optional Colab fallback: upload the assignment CSV
# ============================================================

if selected_assignment_path is None:
    try:
        from google.colab import files

        print(
            "Please upload the selected-Raphe assignment CSV from the previous run "
            "(must contain bodyId and cluster columns)."
        )
        uploaded = files.upload()

        if not uploaded:
            raise RuntimeError("No file uploaded.")

        uploaded_name = next(iter(uploaded.keys()))
        selected_assignment_path = Path(uploaded_name)
        print("Using uploaded file:", selected_assignment_path)

    except ImportError:
        print(
            "Not running in Google Colab. Set CLUSTER_ASSIGNMENTS_PATH manually "
            "in the previous cell."
        )

In [ ]:
# ============================================================
# 9. Extract and print selected-Raphe cluster 7
# ============================================================

if selected_assignment_path is None:
    raise RuntimeError(
        "No assignment table available. Set CLUSTER_ASSIGNMENTS_PATH or upload a CSV."
    )

assignments = pd.read_csv(selected_assignment_path)

required_cols = {"bodyId", "cluster"}
missing_cols = required_cols - set(assignments.columns)
if missing_cols:
    raise RuntimeError(
        f"Assignment table is missing required columns: {sorted(missing_cols)}. "
        f"Columns found: {list(assignments.columns)}"
    )

assignments = assignments.copy()
assignments["bodyId"] = assignments["bodyId"].astype(int)
assignments["cluster"] = assignments["cluster"].astype(int)

cluster7_assignments = (
    assignments.loc[
        assignments["cluster"] == int(SELECTED_RAPHE_CLUSTER_ID)
    ]
    .copy()
    .sort_values("bodyId")
    .reset_index(drop=True)
)

cluster7_body_ids = cluster7_assignments["bodyId"].astype(int).tolist()

print("Assignment table:", selected_assignment_path)
print("Total clustered cells:", len(assignments))
print("Available cluster IDs:", sorted(assignments["cluster"].unique().tolist()))
print(
    f"\nSelected-Raphe cluster {SELECTED_RAPHE_CLUSTER_ID}: "
    f"{len(cluster7_body_ids)} neurons"
)

display(cluster7_assignments)

print("\nCluster 7 body IDs:")
print(cluster7_body_ids)

cluster7_out = CACHE_DIR / "selected_raphe_cluster7_bodyIds.csv"
pd.DataFrame({"bodyId": cluster7_body_ids}).to_csv(cluster7_out, index=False)
print("\nSaved:", cluster7_out)

In [ ]:
# ============================================================
# 10. Plot selected-Raphe cluster 7 skeletons
# ============================================================

cluster7_skeleton_figure, cluster7_skeleton_ids_plotted = plot_skeletons_3d(
    cluster7_body_ids,
    title=f"Selected-Raphe morphology cluster {SELECTED_RAPHE_CLUSTER_ID}",
    color=CLUSTER7_COLOR,
    line_width=CLUSTER7_LINE_WIDTH,
    max_neurons=MAX_CLUSTER7_SKELETONS_TO_PLOT,
)

## Suggested student questions

- Why can a neuron that “touches Raphe superior” have a soma far from the Raphe?
- What morphological features appear shared within cluster 7?
- Do cluster-7 neurons occupy similar trajectories or projection territories?
- How would you compare morphology clusters with input/output connectivity?
- What additional evidence would be needed to infer serotonergic identity?